<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabianalexisbernabecruz-code/portafolio-actuaria-unam/blob/main/13_Credit_Risk_Balancing_Interpretability.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam)

</div>



<font face="Times New Roman">

 <img src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg" width="150" height="170" align="left"/>

  <img src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg" width="150" height="170" align="right"/>

<h5 align="center"><i> </i></h5>
<h1 align="center"><b>Riesgo Crediticio</b></h1>
</font>
<font face="Times New Roman">

---
---

<h2 align="center"><i>Balanceo e Interpretabilidad en Modelos de Crédito</i></h2>
<h5 align="center"><i>Técnicas avanzadas de balanceo de clases (SMOTE) e interpretabilidad global y local (SHAP).</i></h5>

---
---

<h5 align="center"><i>Universidad Nacional Autónoma de México</i></h5>
<h5 align="center"><i>Facultad de Ciencias — Actuaría</i></h5>
<h5 align="center"><i>Materia: Seminario de Estadística Matemática y Computacional</i></h5>

---
---

<font color="Black" face="Times New Roman">
<h5 align="center"><i>Bernabe Cruz Fabian Alexis</i></h5>

</font>
</font>

<font face="Times New Roman">

### **1. Librerías y Configuración**

</font>

In [ ]:
!pip install lime
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import lime.lime_tabular
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
import warnings
from scipy.stats import rankdata
import kagglehub
import os
import time
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import friedmanchisquare, shapiro
try:
    import scikit_posthocs as sp
except ImportError:
    !pip install scikit-posthocs
    import scikit_posthocs as sp
from sklearn.metrics import roc_auc_score
from scipy.stats import shapiro, friedmanchisquare

warnings.filterwarnings('ignore')
shap.initjs()

<font face="Times New Roman">

### **2. Carga, Preprocesamiento y Prevención de Fuga de Datos (Data Leakage)**
1. Ingesta del archivo original de Credit Risk (German Credit Risk).
2. Limpieza de datos nulos (asignándolos como 'unknown').
3. Transformación de la variable objetivo binaria (0 = 'good', 1 = 'bad') y Codificación (OHE) de características categóricas.
4. **Prevención de Data Leakage:** Se realiza la partición de prueba (`train_test_split`) de manera temprana.
5. **Estandarización Rigurosa:** Las variables numéricas continuas se escalan ajustando y conociendo estadísticamente el conjunto de entrenamiento de forma aislada, proyectando luego estas distribuciones maestras al conjunto de prueba para erradicar el aprendizaje anticipado espurio.
</font>

In [ ]:
path = kagglehub.dataset_download("kabure/german-credit-data-with-risk")

csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    csv_file_name = csv_files[0]
    full_csv_path = os.path.join(path, csv_file_name)
    df = pd.read_csv(full_csv_path, sep=',')
    print("Datos cargados exitosamente desde kagglehub.")
    display(df.head())
else:
    print(f"No se encontró ningún archivo CSV en el directorio: {path}")


In [ ]:
# 1. Manejo de nulos
df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

# 2. Codificación de Variable Objetivo
df['Risk'] = df['Risk'].map({'bad': 1, 'good': 0})

# 3. Codificación de Variables Categóricas (OHE)
X = df.drop(columns=['Risk'])
y = df['Risk']

categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 4. Partición Temprana (Prevención de Data Leakage)
SEMILLA = 42
TEST_SIZE = 0.3
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded, y, test_size=TEST_SIZE, stratify=y, random_state=SEMILLA
)

# 5. Escalamiento de variables numéricas SIN Fuga de Datos
numeric_cols = ['Age', 'Credit amount', 'Duration']
scaler = StandardScaler()

# Fit solo en Train, transform en ambos
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

X_train[numeric_cols] = scaler.fit_transform(X_train_raw[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test_raw[numeric_cols])

print(f"Dimensiones de X_train escalado rigurosamente: {X_train.shape}")
print(f"Dimensiones de X_test preparado: {X_test.shape}")

<font face="Times New Roman">

### **3. Configuración Experimental y Tamaño de Muestra (Z) Constante**
El diseño experimental evalúa cómo se degradan las métricas SRA y CV para diferentes niveles de desbalance sintético ($1\%, 2.5\%, \dots, 50\%$) evaluado con un solo nivel de profundidad $d=1$ tras aplicar técnicas de balanceo (RUS/ROS) y un escenario Base.

Para garantizar el rigor empírico y asilar el verdadero efecto que el submuestreo de clases ocasiona, se determinó **fijar de manera inamovible el tamaño de la muestra ($Z$)** durante todas las inducciones sintéticas de diferentes franjas. De esta manera, se excluye el efecto de pérdida o adición de poder estadístico originado por mera de reducción volumétrica. Se acotó orgánicamente un **$Z = 420$** para la partición Target.

**Estructura Constante del Dataset Inducido a $Z=420$:**

| Default Rate | Defaults (Ej. Malos) | Non-Defaults (Ej. Buenos) | Tamaño Total Z |
| :---: | :---: | :---: | :---: |
| 1% | 4 | 416 | 420 |
| 2.5% | 10 | 410 | 420 |
| 5% | 21 | 399 | 420 |
| 10% | 42 | 378 | 420 |
| 15% | 63 | 357 | 420 |
| 20% | 84 | 336 | 420 |
| 25% | 105 | 315 | 420 |
| 30% | 126 | 294 | 420 |
| 35% | 147 | 273 | 420 |
| 40% | 168 | 252 | 420 |
| 45% | 189 | 231 | 420 |
| 50% | 210 | 210 | 420 |

Establecemos el Target Set ($T$) de 50 instancias aleatorias de la partición de prueba (Test set) manteniendo la distribución original representativa pre-balanceos.
</font>

In [ ]:
# Variables globales metodológicas experimentales
TARGET_SET_SIZE = 50
REPETICIONES = 10

# Target Set T fijo para comparar estabilidad local
target_indices = np.random.RandomState(SEMILLA).choice(X_test.index, size=TARGET_SET_SIZE, replace=False)
X_target = X_test.loc[target_indices]
y_target = y_test.loc[target_indices]

def inject_imbalance(X, y, target_ratio, random_state=SEMILLA):
    """
    Inyecta artificialmente un nivel de desbalance (minority ratio) submuestreando
    pero asegurando que el tamaño total (Z) del conjunto de datos devuelto se
    mantenga estrictamente fijo para excluir efectos por volumen,
    replicando textualmente el marco de validación.
    """
    minority_class = 1
    majority_class = 0

    idx_minority = y[y == minority_class].index
    idx_majority = y[y == majority_class].index

    # Fijamos el Z al doble máximo balanceado con la clase minoritaria real
    Z = 2 * len(idx_minority)

    # Cantidades inamovibles bajo Z:
    n_minority_target = max(1, int(round(Z * target_ratio)))
    n_majority_target = Z - n_minority_target

    rs = np.random.RandomState(random_state)

    idx_minority_sampled = rs.choice(idx_minority, size=n_minority_target, replace=False)
    idx_majority_sampled = rs.choice(idx_majority, size=n_majority_target, replace=False)

    final_idx = np.concatenate([idx_majority_sampled, idx_minority_sampled])
    return X.loc[final_idx], y.loc[final_idx]

def calcular_sra_d1(listas_rangos):
    """
    Calcula SRA evaluado exclusivamente en d=1 (un solo nivel de profundidad).
    """
    P = listas_rangos.shape[1]
    varianzas_elementos = np.var(listas_rangos, axis=0, ddof=1)
    # d = 1
    mascara_Sd1 = np.any(listas_rangos <= 1, axis=0)
    varianzas_Sd1 = varianzas_elementos[mascara_Sd1]
    if len(varianzas_Sd1) > 0 and not np.isnan(np.mean(varianzas_Sd1)):
        return np.sqrt(np.mean(varianzas_Sd1))
    return 0.0

def calculate_cv_avg(attributions):
    """Calcula Coeficiente de Variación promedio (CV) en la magnitud para una instancia."""
    stds = np.std(attributions, axis=0)
    means = np.mean(np.abs(attributions), axis=0)
    cvs = stds / (means + 1e-10)
    return np.mean(cvs)

<font face="Times New Roman">

### **4. Ciclo de Experimentos y Degradación de Estabilidad**
Se itera a través de los niveles teóricos de desbalanceo $1\%, 2.5\%, 5\%, 10\%, 15\% \dots 50\%$. Para cada franja, se indujo el sesgo en el conjunto de entrenamiento y se evaluó el desempeño y volatilidad con iteraciones Bootstrap (`REPETICIONES = 10`) utilizando el algoritmo original (Base), RUS y ROS.

> **NOTA:** El tiempo de ejecución es alto porque se ensamblan múltiples modelos XGBoost y RF por cada nivel de inyección sintética (Tiempo estimado: ~ 100 minutos).
</font>

In [ ]:
escenarios = ['Base', 'RUS', 'ROS']
niveles_desbalance = [0.01, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

# Estructura: metricas[nivel][escenario] = valor
resultados_cv_shap_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_cv_lime_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
sra_d1_shap_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
sra_d1_lime_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_auc_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_f1_xgb = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}

# Nuevas variables para Random Forest
resultados_cv_shap_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_cv_lime_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
sra_d1_shap_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
sra_d1_lime_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_auc_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}
resultados_f1_rf = {lvl: {esc: [] for esc in escenarios} for lvl in niveles_desbalance}

def extract_lime_weights(explanation, feature_names):
    lime_dict = dict(explanation.as_list())
    pesos_ordenados = []
    for col in feature_names:
        peso = 0.0
        for lime_feat, eval_w in lime_dict.items():
            if col in lime_feat:
                peso = eval_w
                break
        pesos_ordenados.append(peso)
    return np.array(pesos_ordenados)

print("Iniciando Evaluación con XGBoost y Random Forest...")
start_time = time.time()

for lvl in niveles_desbalance:
    print(f"\n=== Evaluando nivel de desbalanceo minoritario: {lvl * 100:.1f}% ===")
    X_train_imb, y_train_imb = inject_imbalance(X_train, y_train, target_ratio=lvl)

    for esc in escenarios:
        print(f"  > Escenario: {esc}")

        all_shap_values_xgb = []
        all_lime_values_xgb = []
        all_auc_xgb_list = []
        all_f1_xgb_list = []

        all_shap_values_rf = []
        all_lime_values_rf = []
        all_auc_rf_list = []
        all_f1_rf_list = []

        for r in range(REPETICIONES):
            semilla_r = SEMILLA + r

            # 1. Remuestreo
            if esc == 'Base':
                Xr, yr = X_train_imb, y_train_imb
            elif esc == 'RUS':
                n_minority = sum(y_train_imb == 1)
                if n_minority > 0:
                   try:
                       Xr, yr = RandomUnderSampler(random_state=semilla_r).fit_resample(X_train_imb, y_train_imb)
                   except ValueError:
                       Xr, yr = X_train_imb, y_train_imb
                else:
                   Xr, yr = X_train_imb, y_train_imb
            elif esc == 'ROS':
                try:
                    Xr, yr = RandomOverSampler(random_state=semilla_r).fit_resample(X_train_imb, y_train_imb)
                except ValueError:
                    Xr, yr = X_train_imb, y_train_imb

            if len(np.unique(yr)) <= 1:
                continue

            # --- MODELO 1: XGBoost ---
            model_xgb = xgb.XGBClassifier(random_state=semilla_r, eval_metric='logloss', subsample=0.8, colsample_bytree=0.8)
            model_xgb.fit(Xr, yr)
            y_pred_xgb = model_xgb.predict(X_test)
            y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
            all_f1_xgb_list.append(f1_score(y_test, y_pred_xgb, zero_division=0))
            all_auc_xgb_list.append(roc_auc_score(y_test, y_proba_xgb))

            explainer_shap_xgb = shap.TreeExplainer(model_xgb)
            all_shap_values_xgb.append(explainer_shap_xgb.shap_values(X_target))

            explainer_lime_xgb = lime.lime_tabular.LimeTabularExplainer(np.array(Xr), feature_names=list(X_train.columns), class_names=['Good', 'Bad'], mode='classification', random_state=semilla_r)
            lime_v_r_xgb = []
            for i in range(TARGET_SET_SIZE):
                exp = explainer_lime_xgb.explain_instance(X_target.iloc[i], model_xgb.predict_proba, num_features=len(X_train.columns))
                lime_v_r_xgb.append(extract_lime_weights(exp, X_train.columns))
            all_lime_values_xgb.append(np.array(lime_v_r_xgb))

            # --- MODELO 2: Random Forest ---
            model_rf = RandomForestClassifier(random_state=semilla_r, n_estimators=100)
            model_rf.fit(Xr, yr)
            y_pred_rf = model_rf.predict(X_test)
            y_proba_rf = model_rf.predict_proba(X_test)[:, 1]
            all_f1_rf_list.append(f1_score(y_test, y_pred_rf, zero_division=0))
            all_auc_rf_list.append(roc_auc_score(y_test, y_proba_rf))

            explainer_shap_rf = shap.TreeExplainer(model_rf)
            # TreeExplainer for RF returns a list of shape values (one per class), we take [:,:,1]
            shap_vals_rf = explainer_shap_rf.shap_values(X_target)
            if isinstance(shap_vals_rf, list):
                shap_vals_rf = shap_vals_rf[1]
            all_shap_values_rf.append(shap_vals_rf)

            explainer_lime_rf = lime.lime_tabular.LimeTabularExplainer(np.array(Xr), feature_names=list(X_train.columns), class_names=['Good', 'Bad'], mode='classification', random_state=semilla_r)
            lime_v_r_rf = []
            for i in range(TARGET_SET_SIZE):
                exp = explainer_lime_rf.explain_instance(X_target.iloc[i], model_rf.predict_proba, num_features=len(X_train.columns))
                lime_v_r_rf.append(extract_lime_weights(exp, X_train.columns))
            all_lime_values_rf.append(np.array(lime_v_r_rf))

        # --- Calcular métricas (SRA y CV) ---
        inst_cvs_shap_xgb, inst_cvs_lime_xgb = [], []
        sra_por_inst_shap_xgb, sra_por_inst_lime_xgb = [], []

        inst_cvs_shap_rf, inst_cvs_lime_rf = [], []
        sra_por_inst_shap_rf, sra_por_inst_lime_rf = [], []

        for i in range(TARGET_SET_SIZE):
            # XGB
            s_attr_xgb = np.array([all_shap_values_xgb[r][i] for r in range(len(all_shap_values_xgb))])
            l_attr_xgb = np.array([all_lime_values_xgb[r][i] for r in range(len(all_lime_values_xgb))])
            if len(s_attr_xgb) > 0:
                inst_cvs_shap_xgb.append(calculate_cv_avg(s_attr_xgb))
                inst_cvs_lime_xgb.append(calculate_cv_avg(l_attr_xgb))
                sra_por_inst_shap_xgb.append(calcular_sra_d1(np.array([rankdata(-np.abs(s_attr_xgb[r]), method='average') for r in range(len(s_attr_xgb))])))
                sra_por_inst_lime_xgb.append(calcular_sra_d1(np.array([rankdata(-np.abs(l_attr_xgb[r]), method='average') for r in range(len(l_attr_xgb))])))

            # RF
            s_attr_rf = np.array([all_shap_values_rf[r][i] for r in range(len(all_shap_values_rf))])
            l_attr_rf = np.array([all_lime_values_rf[r][i] for r in range(len(all_lime_values_rf))])
            if len(s_attr_rf) > 0:
                inst_cvs_shap_rf.append(calculate_cv_avg(s_attr_rf))
                inst_cvs_lime_rf.append(calculate_cv_avg(l_attr_rf))
                sra_por_inst_shap_rf.append(calcular_sra_d1(np.array([rankdata(-np.abs(s_attr_rf[r]), method='average') for r in range(len(s_attr_rf))])))
                sra_por_inst_lime_rf.append(calcular_sra_d1(np.array([rankdata(-np.abs(l_attr_rf[r]), method='average') for r in range(len(l_attr_rf))])))

        # Guardar en diccionarios
        resultados_cv_shap_xgb[lvl][esc] = np.nanmean(inst_cvs_shap_xgb) if inst_cvs_shap_xgb else np.nan
        resultados_cv_lime_xgb[lvl][esc] = np.nanmean(inst_cvs_lime_xgb) if inst_cvs_lime_xgb else np.nan
        sra_d1_shap_xgb[lvl][esc] = np.nanmean(sra_por_inst_shap_xgb) if sra_por_inst_shap_xgb else np.nan
        sra_d1_lime_xgb[lvl][esc] = np.nanmean(sra_por_inst_lime_xgb) if sra_por_inst_lime_xgb else np.nan
        resultados_f1_xgb[lvl][esc] = np.nanmean(all_f1_xgb_list) if all_f1_xgb_list else np.nan
        resultados_auc_xgb[lvl][esc] = np.nanmean(all_auc_xgb_list) if all_auc_xgb_list else np.nan

        resultados_cv_shap_rf[lvl][esc] = np.nanmean(inst_cvs_shap_rf) if inst_cvs_shap_rf else np.nan
        resultados_cv_lime_rf[lvl][esc] = np.nanmean(inst_cvs_lime_rf) if inst_cvs_lime_rf else np.nan
        sra_d1_shap_rf[lvl][esc] = np.nanmean(sra_por_inst_shap_rf) if sra_por_inst_shap_rf else np.nan
        sra_d1_lime_rf[lvl][esc] = np.nanmean(sra_por_inst_lime_rf) if sra_por_inst_lime_rf else np.nan
        resultados_f1_rf[lvl][esc] = np.nanmean(all_f1_rf_list) if all_f1_rf_list else np.nan
        resultados_auc_rf[lvl][esc] = np.nanmean(all_auc_rf_list) if all_auc_rf_list else np.nan

print(f"\nEvaluación completada exitosamente en {(time.time() - start_time)/60:.2f} minutos.")



<font face="Times New Roman">

### **5. Análisis de Degradación Numérica (Muestra Consolidada)**
Resumen de los niveles evaluados y el impacto en SRA (a un nivel $d=1$) y CV.
</font>

In [ ]:
resumen_global = []

for lvl in niveles_desbalance:
    for esc in escenarios:
        resumen_global.append({
            'Nivel Imbalance (%)': lvl * 100,
            'Escenario': esc,
            'SHAP CV (XGB)': resultados_cv_shap_xgb[lvl][esc],
            'LIME CV (XGB)': resultados_cv_lime_xgb[lvl][esc],
            'SHAP SRA (XGB)': sra_d1_shap_xgb[lvl][esc],
            'LIME SRA (XGB)': sra_d1_lime_xgb[lvl][esc],
            'F1-Score (XGB)': resultados_f1_xgb[lvl][esc],
            'AUC (XGB)': resultados_auc_xgb[lvl][esc],
            'SHAP CV (RF)': resultados_cv_shap_rf[lvl][esc],
            'LIME CV (RF)': resultados_cv_lime_rf[lvl][esc],
            'SHAP SRA (RF)': sra_d1_shap_rf[lvl][esc],
            'LIME SRA (RF)': sra_d1_lime_rf[lvl][esc],
            'F1-Score (RF)': resultados_f1_rf[lvl][esc],
            'AUC (RF)': resultados_auc_rf[lvl][esc]
        })

df_resumen = pd.DataFrame(resumen_global)
display(df_resumen.head())
print('Resumen estadístico de las métricas (XGBoost y RF):')
display(df_resumen.groupby('Escenario').mean())



<font face="Times New Roman">

### **6. Visualización de Resultados: CV y SRA**
Desplegamos los boxplots y gráficos de línea para contrastar de forma visual la volatilidad y evaluar nuestra hipótesis de si **RUS/ROS introducen inestabilidad** natural aportada además por la naturaleza de LIME vs SHAP.

</font>

In [ ]:
import os
FIG_DIR = 'figures_german_credit'
os.makedirs(FIG_DIR, exist_ok=True)

colors = {'Base': 'gray', 'RUS': 'red', 'ROS': 'green'}
markers_xgb = {'Base': 'o', 'RUS': 's', 'ROS': '^'}
markers_rf = {'Base': 'x', 'RUS': '+', 'ROS': 'd'}

plots = [
    ('SHAP CV (XGB)', 'SHAP CV (RF)', 'Degradación CV de SHAP (German Credit)', 'Promedio CV (Menor = Más estable)', 'shap_cv'),
    ('LIME CV (XGB)', 'LIME CV (RF)', 'Degradación CV de LIME (German Credit)', 'Promedio CV (Menor = Más estable)', 'lime_cv'),
    ('SHAP SRA (XGB)', 'SHAP SRA (RF)', 'Degradación SRA (d=1) de SHAP (German Credit)', 'SRA d=1 (Menor = Más estable)', 'shap_sra'),
    ('LIME SRA (XGB)', 'LIME SRA (RF)', 'Degradación SRA (d=1) de LIME (German Credit)', 'SRA d=1 (Menor = Más estable)', 'lime_sra'),
    ('AUC (XGB)', 'AUC (RF)', 'Capacidad Predictiva: AUC (German Credit)', 'AUC (Mayor = Mejor)', 'auc'),
    ('F1-Score (XGB)', 'F1-Score (RF)', 'Capacidad Predictiva: F1-Score (German Credit)', 'F1-Score (Mayor = Mejor)', 'f1'),
]


# Consolidación gráfica: Estandarizar ejes Y para comparabilidad
max_cv = df_resumen[['SHAP CV (XGB)', 'SHAP CV (RF)', 'LIME CV (XGB)', 'LIME CV (RF)']].max().max() * 1.1
max_sra = df_resumen[['SHAP SRA (XGB)', 'SHAP SRA (RF)', 'LIME SRA (XGB)', 'LIME SRA (RF)']].max().max() * 1.1

for col_xgb, col_rf, title, ylabel, fname in plots:

    fig, ax = plt.subplots(figsize=(10, 6))
    for esc in escenarios:
        sub_df = df_resumen[df_resumen['Escenario'] == esc]
        x_axis = sub_df['Nivel Imbalance (%)']
        ax.plot(x_axis, sub_df[col_xgb], label=f"{esc} (XGB)", color=colors[esc], marker=markers_xgb[esc], linestyle='-')
        ax.plot(x_axis, sub_df[col_rf],  label=f"{esc} (RF)",  color=colors[esc], marker=markers_rf[esc],  linestyle='--')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Nivel de Desbalanceo (%)', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(x_axis)
    ax.set_xticklabels([f"{x:.1f}%" for x in x_axis], rotation=45)
    if 'cv' in fname:
        ax.set_ylim(0, max_cv)
    elif 'sra' in fname:
        ax.set_ylim(0, max_sra)
    ax.legend(title='Escenario', fontsize='small', loc='best')
    ax.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    save_path = os.path.join(FIG_DIR, f'german_{fname}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardada: {save_path}")


<font face="Times New Roman">

### **7. Interpretación de Efectos por Desbalanceo**

Basado en las curvas de degradación generadas bajo distintos espectros de desbalance (desde un extremo de 1% hasta un ideal 50%), se pueden observar contundentemente los siguientes fenómenos en las métricas de variabilidad estocástica local (SRA con $d=1$, considerando solo el rasgo de decisión primordial, y CV):

1. **La degradación acelerada del Submuestreo (RUS):** Conforme disminuimos la representación natural porcentual de la clase minoritaria acercándonos al 1%-5%, descartar la valiosa información de la clase mayoritaria (a través de RUS) para forzar un balance desencadena una inestabilidad geométrica inmensa en modelos locales (SHAP y LIME).
2. **Robustez ante Remuestreo por Extensión (ROS):** En marcado contraste a RUS, mantener las fronteras mediante duplicación sintética de ROS no fragmenta bruscamente la topología local de decisión del algoritmo. De hecho, a través de todos los abanicos porcentuales (2.5% a 50%), **ROS** muestra curvas de Coeficiente de Variación estables que imitan el comportamiento base del modelo XGBoost sesgado sin preprocesadores en casi todos sus percentiles de evaluación.
3. **Fallas inherentes en LIME frente a variabilidad:** Sin importar el nivel exacto provisto entre $1\%$ o $45\%$, y de forma notable con RUS pero también con intervenciones artificiales de desbalanceo, LIME manifiesta una severa volatilidad al permutar las variables locales en la profundidad única de SRA ($d=1$). Su naturaleza de aproximación Ridge mediante muestras alteradas es incapaz de lidiar con submuestreos aleatorios y muestra dispersiones inaceptables para auditar el perfil crediticio, en un rango donde la atribución SHAP sufre menor perturbación relativa por tener una axiomática de valor estricto de juego colaborativo.

---

### **8. Conclusión General del Experimento**

La investigación confirma plenamente la hipótesis central postulada: **las métricas de consistencia SRA (acuerdo más relevante 1-D) y el Coeficiente de Variación se degradan severamente en esquemas de inyección sintética estresa de desbalance extremo ($\sim 1\% - 5\%$ de clase minoritaria) cuando van acompañados de algoritmos sub-óóptimos de balanceo clásico**, como es el caso de RUS.

El impacto sobre XAI obedece a razones sistémicas: el uso indiscriminado de **Random Under-Sampling (RUS)** trae consigo un coste severo sobre la homogeneidad visual de las razones otorgadas al cliente por XGBoost. En este contexto, **LIME demuestra ser empíricamente más vulnerable e inestable** frente a las alteraciones del conjunto de datos comparado con SHAP, mostrando una dispersión inaceptable incluso bajo metodologías de remuestreo sintético, lo que evidencia que su uso local no soporta bien escenarios escasos. Esto es crítico porque frente a regulaciones modernas, un modelo que carece de consistencia local al explicar exactamente la misma solicitud de crédito bajo pequeñas perturbaciones matemáticas incurre en un riesgo fiduciario alto y es inválido bajo escrutinio de los entes reguladores.

Se destaca que **Random Over-Sampling (ROS)** junto a algoritmos enraizados al marco de valores exactos como **SHAP** brindan la ruta mitigante idónea frente a la inestabilidad y dispersión metodológica inherente de **LIME**, posibilitando conservar métricas de SRA próximas a líneas base aun cuando nos desplacemos en espectros de extremo desequilibrio natural en el mercado.
</font>

<font face="Times New Roman">

## 4. Análisis Estadístico Adicional (German Credit Data)

Para dotar de rigor estadístico a los hallazgos observados en los escenarios de *German Credit*, debemos verificar primero el cumplimiento de los supuestos para la selección adecuada de la prueba estadística:
1. **Prueba de Shapiro-Wilk**: Para verificar si los datos (CV de SHAP para **XGBoost**) siguen una distribución normal. Si se cumple, correspondería un ANOVA de medidas repetidas; de lo contrario, se justifica usar Friedman.
2. **Prueba de Friedman**: Una alternativa no paramétrica a ANOVA de medidas repetidas, utilizada cuando se viola el supuesto de normalidad, para validar si existen diferencias significativas en la estabilidad de XGBoost a través de los múltiples escenarios.
3. **Prueba Post-hoc de Nemenyi**: En caso de que Friedman rechace la hipótesis nula, esta prueba nos indicará exactamente entre qué pares de escenarios (Base, RUS, ROS) radican las diferencias significativas.

</font>


In [ ]:
import numpy as np

print("=== Análisis Estadístico XGBoost (German Credit) ===")
niveles = list(resultados_cv_shap_xgb.keys())
escenarios_list = ['Base', 'RUS', 'ROS']

matriz_shap = []
for lvl in niveles:
    fila = [resultados_cv_shap_xgb[lvl][esc] for esc in escenarios_list]
    matriz_shap.append(fila)
matriz_shap = np.array(matriz_shap)

mask = ~np.isnan(matriz_shap).any(axis=1)
matriz_shap = matriz_shap[mask]

if len(matriz_shap) > 2:
    print("1. Comprobación de supuestos (Normalidad con Shapiro-Wilk):")
    p_norm_base = shapiro(matriz_shap[:, 0])[1]
    p_norm_rus = shapiro(matriz_shap[:, 1])[1]
    p_norm_ros = shapiro(matriz_shap[:, 2])[1]
    print(f"   - Base p-value: {p_norm_base:.4e}")
    print(f"   - RUS  p-value: {p_norm_rus:.4e}")
    print(f"   - ROS  p-value: {p_norm_ros:.4e}")

    if any(p < 0.05 for p in [p_norm_base, p_norm_rus, p_norm_ros]):
        print("   -> Al menos un escenario no sigue una distribución normal (p < 0.05).")
        print("   -> Se justifica el uso de la prueba no paramétrica de Friedman.\n")
    else:
        print("   -> Los datos siguen una distribución normal. Se sugiere ANOVA de medidas repetidas, pero procedemos con Friedman por robustez no paramétrica.\n")

    print("2. Prueba de Friedman:")
    stat, p_value = friedmanchisquare(matriz_shap[:, 0], matriz_shap[:, 1], matriz_shap[:, 2])
    print(f"   -> Friedman Test (SHAP CV) -> Statistic: {stat:.4f}, p-value: {p_value:.4e}")

    if p_value < 0.05:
        print("\n3. Existen diferencias significativas entre los escenarios. Aplicando Nemenyi Post-hoc...")
        nemenyi_res = sp.posthoc_nemenyi_friedman(matriz_shap)
        nemenyi_res.columns = escenarios_list
        nemenyi_res.index = escenarios_list
        display(nemenyi_res)
    else:
        print("\nNo se encontraron diferencias estadísticamente significativas con 95% de confianza.")
else:
    print("Datos insuficientes para la prueba de Friedman.")


<font face="Times New Roman">

**Interpretación de Resultados (German Credit):**
Las pruebas estadísticas ejecutadas sobre las métricas de estabilidad evidencian que se incumple el supuesto de normalidad (según Shapiro-Wilk), validando el uso de pruebas no paramétricas.

La **prueba de Friedman arroja un *p-value* inferior al nivel de significancia $\alpha = 0.05$**. Esto confirma de manera estadística que **el mecanismo de remuestreo (RUS, ROS) altera de forma significativa la estabilidad explicativa** en comparación con el escenario Base. Adicionalmente, los contrastes post-hoc de Nemenyi respaldan que la discrepancia y degradación más severa es inducida sistemáticamente por el algoritmo RUS.

</font>

<font face="Times New Roman">

---
---

# 5. Análisis de Robustez del Experimento

Para confirmar que los hallazgos previos no son un artefacto exclusivo de la base de datos *German Credit* , replicamos el marco de validación íntegro en un entorno de **Robustez**. Se integrará un nuevo conjunto de datos, una nueva arquitectura de caja negra, y se incorpora.

</font>


<font face="Times New Roman">

### 5.1 Carga y Limpieza de Datos Adicionales (Home Equity - HMEQ)

Utilizaremos el conjunto de datos **Home Equity (HMEQ)**. Para este conjunto, la variable objetivo ya viene definida binariamente en la columna `BAD` (1 = default, 0 = pagado). Se aplicará una estrategia de imputación por la mediana para numéricos y categorización de *Unknown* para nulos cualitativos.

</font>


In [ ]:
print("Descargando dataset HMEQ para robustez...")
path = kagglehub.dataset_download("ajay1735/hmeq-data")
df_hmeq = pd.read_csv(os.path.join(path, "hmeq.csv"))
print("Dataset HMEQ cargado. Shape:", df_hmeq.shape)

num_cols = df_hmeq.select_dtypes(include=['number']).columns
df_hmeq[num_cols] = df_hmeq[num_cols].fillna(df_hmeq[num_cols].median())
df_hmeq['REASON'] = df_hmeq['REASON'].fillna('Unknown')
df_hmeq['JOB'] = df_hmeq['JOB'].fillna('Unknown')

X_rob = df_hmeq.drop(columns=['BAD'])
y_rob = df_hmeq['BAD']

cat_cols_rob = X_rob.select_dtypes(include=['object']).columns
X_rob_encoded = pd.get_dummies(X_rob, columns=cat_cols_rob, drop_first=True)

SEMILLA_ROB = 101
X_train_raw_rob, X_test_raw_rob, y_train_rob, y_test_rob = train_test_split(
    X_rob_encoded, y_rob, test_size=0.3, stratify=y_rob, random_state=SEMILLA_ROB
)

scaler_rob = StandardScaler()
X_train_rob = X_train_raw_rob.copy()
X_test_rob = X_test_raw_rob.copy()
num_encoded_cols = X_train_raw_rob.select_dtypes(include=['number']).columns

X_train_rob[num_encoded_cols] = scaler_rob.fit_transform(X_train_raw_rob[num_encoded_cols])
X_test_rob[num_encoded_cols] = scaler_rob.transform(X_test_raw_rob[num_encoded_cols])

print(f"Dimensiones X_train HMEQ: {X_train_rob.shape}")


<font face="Times New Roman">

### 5.2 Metodología y Variables Globales para Robustez

Se mantiene la evaluación simultánea de **XGBoost** y **Random Forest Classifier**.
Definiremos la malla experimental completa dictada metodológicamente: $u \in \{1\%, 2.5\%, 5\%, 10\%, ..., 50\%\}$ con $B=10$ repeticiones para aislar correctamente la varianza.

</font>


In [ ]:
TARGET_SET_SIZE_ROB = 50
target_indices_rob = np.random.RandomState(SEMILLA_ROB).choice(X_test_rob.index, size=TARGET_SET_SIZE_ROB, replace=False)
X_target_rob = X_test_rob.loc[target_indices_rob]
y_target_rob = y_test_rob.loc[target_indices_rob]

def inject_imbalance_rob(X, y, target_ratio, random_state):
    min_c, maj_c = 1, 0
    idx_min = y[y == min_c].index
    idx_maj = y[y == maj_c].index
    Z = 2 * len(idx_min)
    n_min = max(1, int(round(Z * target_ratio)))
    n_maj = Z - n_min
    rs = np.random.RandomState(random_state)
    idx_min_sampled = rs.choice(idx_min, size=n_min, replace=False)
    idx_maj_sampled = rs.choice(idx_maj, size=n_maj, replace=False)
    final_idx = np.concatenate([idx_maj_sampled, idx_min_sampled])
    return X.loc[final_idx], y.loc[final_idx]

def calculate_cv_avg_rob(attributions):
    stds = np.std(attributions, axis=0)
    means = np.mean(np.abs(attributions), axis=0)
    return np.mean(stds / (means + 1e-10))


escenarios_rob = ['Base', 'RUS', 'ROS']
niveles_desbalance_rob = [0.01, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
REPETICIONES_ROB = 10


<font face="Times New Roman">

### 5.3 Bucle Principal de Evaluación (HMEQ)

En este bloque iterativo, para cada escenario de desbalanceo de la malla extendida (1% al 50%), instanciamos y entrenamos tanto XGBoost como Random Forest. A cada modelo se le extraen las explicaciones SHAP y LIME, y se calculan las métricas CV (Estabilidad) y SRA (d=1) (Concordancia de rangos), espejando el diseño metodológico del experimento principal

</font>


In [ ]:
cv_shap_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
cv_shap_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
cv_lime_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
cv_lime_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
sra_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
sra_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
sra_lime_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
sra_lime_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
auc_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
auc_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
f1_xgb_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}
f1_rf_rob = {lvl: {esc: [] for esc in escenarios_rob} for lvl in niveles_desbalance_rob}

def extract_lime_weights_rob(explanation, feature_names):
    lime_dict = dict(explanation.as_list())
    pesos = []
    for col in feature_names:
        peso = 0.0
        for lime_feat, eval_w in lime_dict.items():
            if col in lime_feat:
                peso = eval_w
                break
        pesos.append(peso)
    return np.array(pesos)

print("Iniciando Bucle de Robustez (HMEQ + XGBoost + Random Forest + LIME)...")
print("El diseño metodológico usa la malla completa (12 niveles) con B=10 repeticiones.")
st_rob = time.time()

for lvl in niveles_desbalance_rob:
    print(f"\n=== Evaluando nivel de desbalanceo (HMEQ): {lvl * 100:.1f}% ===")
    X_tr_imb, y_tr_imb = inject_imbalance_rob(X_train_rob, y_train_rob, lvl, SEMILLA_ROB)
    for esc in escenarios_rob:
        print(f"  > Escenario: {esc}")
        shap_v_xgb_all, shap_v_rf_all = [], []
        lime_v_xgb_all, lime_v_rf_all = [], []
        auc_x_all, auc_r_all = [], []
        f1_x_all, f1_r_all = [], []

        for r in range(REPETICIONES_ROB):
            seed_r = SEMILLA_ROB + r
            if esc == 'Base':
                Xr, yr = X_tr_imb, y_tr_imb
            elif esc == 'RUS':
                n_min = sum(y_tr_imb == 1)
                Xr, yr = (RandomUnderSampler(random_state=seed_r).fit_resample(X_tr_imb, y_tr_imb)
                          if n_min > 0 else (X_tr_imb, y_tr_imb))
            else:
                Xr, yr = RandomOverSampler(random_state=seed_r).fit_resample(X_tr_imb, y_tr_imb)

            if len(np.unique(yr)) <= 1: continue

            # --- XGBoost ---
            m_xgb = xgb.XGBClassifier(random_state=seed_r, eval_metric='logloss', subsample=0.8, colsample_bytree=0.8)
            m_xgb.fit(Xr, yr)
            y_proba_xgb = m_xgb.predict_proba(X_test_rob)[:, 1]
            y_pred_xgb = m_xgb.predict(X_test_rob)
            auc_x_all.append(roc_auc_score(y_test_rob, y_proba_xgb))
            f1_x_all.append(f1_score(y_test_rob, y_pred_xgb, zero_division=0))
            shap_v_xgb_all.append(shap.TreeExplainer(m_xgb).shap_values(X_target_rob))

            explainer_lime_xgb = lime.lime_tabular.LimeTabularExplainer(
                np.array(Xr), feature_names=list(X_train_rob.columns),
                class_names=['Good', 'Bad'], mode='classification', random_state=seed_r)
            lime_r_xgb = []
            for i in range(TARGET_SET_SIZE_ROB):
                exp = explainer_lime_xgb.explain_instance(X_target_rob.iloc[i], m_xgb.predict_proba, num_features=len(X_train_rob.columns))
                lime_r_xgb.append(extract_lime_weights_rob(exp, X_train_rob.columns))
            lime_v_xgb_all.append(np.array(lime_r_xgb))

            # --- Random Forest ---
            m_rf = RandomForestClassifier(random_state=seed_r, n_estimators=100)
            m_rf.fit(Xr, yr)
            y_proba_rf = m_rf.predict_proba(X_test_rob)[:, 1]
            y_pred_rf = m_rf.predict(X_test_rob)
            auc_r_all.append(roc_auc_score(y_test_rob, y_proba_rf))
            f1_r_all.append(f1_score(y_test_rob, y_pred_rf, zero_division=0))
            exp_s_rf = shap.TreeExplainer(m_rf).shap_values(X_target_rob)
            if isinstance(exp_s_rf, list): exp_s_rf = exp_s_rf[1]
            shap_v_rf_all.append(exp_s_rf)

            explainer_lime_rf = lime.lime_tabular.LimeTabularExplainer(
                np.array(Xr), feature_names=list(X_train_rob.columns),
                class_names=['Good', 'Bad'], mode='classification', random_state=seed_r)
            lime_r_rf = []
            for i in range(TARGET_SET_SIZE_ROB):
                exp = explainer_lime_rf.explain_instance(X_target_rob.iloc[i], m_rf.predict_proba, num_features=len(X_train_rob.columns))
                lime_r_rf.append(extract_lime_weights_rob(exp, X_train_rob.columns))
            lime_v_rf_all.append(np.array(lime_r_rf))

        # --- Calcular métricas ---
        inst_cvs_x, inst_cvs_r = [], []
        inst_cv_lime_x, inst_cv_lime_r = [], []
        inst_sra_x, inst_sra_r = [], []
        inst_sra_lime_x, inst_sra_lime_r = [], []

        for i in range(TARGET_SET_SIZE_ROB):
            s_xgb = np.array([shap_v_xgb_all[k][i] for k in range(len(shap_v_xgb_all))])
            l_xgb = np.array([lime_v_xgb_all[k][i] for k in range(len(lime_v_xgb_all))])
            if len(s_xgb) > 1:
                inst_cvs_x.append(calculate_cv_avg_rob(s_xgb))
                inst_cv_lime_x.append(calculate_cv_avg_rob(l_xgb))
                inst_sra_x.append(calcular_sra_d1(np.array([rankdata(-np.abs(s_xgb[k]), method='average') for k in range(len(s_xgb))])))
                inst_sra_lime_x.append(calcular_sra_d1(np.array([rankdata(-np.abs(l_xgb[k]), method='average') for k in range(len(l_xgb))])))

            s_rf = np.array([shap_v_rf_all[k][i] for k in range(len(shap_v_rf_all))])
            l_rf = np.array([lime_v_rf_all[k][i] for k in range(len(lime_v_rf_all))])
            if len(s_rf) > 1:
                inst_cvs_r.append(calculate_cv_avg_rob(s_rf))
                inst_cv_lime_r.append(calculate_cv_avg_rob(l_rf))
                inst_sra_r.append(calcular_sra_d1(np.array([rankdata(-np.abs(s_rf[k]), method='average') for k in range(len(s_rf))])))
                inst_sra_lime_r.append(calcular_sra_d1(np.array([rankdata(-np.abs(l_rf[k]), method='average') for k in range(len(l_rf))])))

        cv_shap_xgb_rob[lvl][esc] = np.nanmean(inst_cvs_x) if inst_cvs_x else np.nan
        cv_shap_rf_rob[lvl][esc] = np.nanmean(inst_cvs_r) if inst_cvs_r else np.nan
        cv_lime_xgb_rob[lvl][esc] = np.nanmean(inst_cv_lime_x) if inst_cv_lime_x else np.nan
        cv_lime_rf_rob[lvl][esc] = np.nanmean(inst_cv_lime_r) if inst_cv_lime_r else np.nan
        sra_xgb_rob[lvl][esc] = np.nanmean(inst_sra_x) if inst_sra_x else np.nan
        sra_rf_rob[lvl][esc] = np.nanmean(inst_sra_r) if inst_sra_r else np.nan
        sra_lime_xgb_rob[lvl][esc] = np.nanmean(inst_sra_lime_x) if inst_sra_lime_x else np.nan
        sra_lime_rf_rob[lvl][esc] = np.nanmean(inst_sra_lime_r) if inst_sra_lime_r else np.nan
        auc_xgb_rob[lvl][esc] = np.nanmean(auc_x_all) if auc_x_all else np.nan
        auc_rf_rob[lvl][esc] = np.nanmean(auc_r_all) if auc_r_all else np.nan
        f1_xgb_rob[lvl][esc] = np.nanmean(f1_x_all) if f1_x_all else np.nan
        f1_rf_rob[lvl][esc] = np.nanmean(f1_r_all) if f1_r_all else np.nan

print(f"Bucle de robustez completado en {(time.time() - st_rob)/60:.2f} mins.")


<font face="Times New Roman">

### 5.4 Consolidación de Resultados

Presentamos la tabla resumen de métricas para la configuración de robustez (HMEQ).

</font>


In [ ]:
df_rob_list = []
for lvl in niveles_desbalance_rob:
    for esc in escenarios_rob:
        df_rob_list.append({
            'Imbalance (%)': lvl * 100,
            'Escenario': esc,
            'SHAP CV (XGB)': cv_shap_xgb_rob[lvl][esc],
            'LIME CV (XGB)': cv_lime_xgb_rob[lvl][esc],
            'SHAP SRA (XGB)': sra_xgb_rob[lvl][esc],
            'LIME SRA (XGB)': sra_lime_xgb_rob[lvl][esc],
            'F1-Score (XGB)': f1_xgb_rob[lvl][esc],
            'AUC (XGB)': auc_xgb_rob[lvl][esc],
            'SHAP CV (RF)': cv_shap_rf_rob[lvl][esc],
            'LIME CV (RF)': cv_lime_rf_rob[lvl][esc],
            'SHAP SRA (RF)': sra_rf_rob[lvl][esc],
            'LIME SRA (RF)': sra_lime_rf_rob[lvl][esc],
            'F1-Score (RF)': f1_rf_rob[lvl][esc],
            'AUC (RF)': auc_rf_rob[lvl][esc]
        })
df_rob = pd.DataFrame(df_rob_list)
display(df_rob.head())
print('Resumen estadístico de las métricas (XGBoost y RF - HMEQ):')
display(df_rob.groupby('Escenario').mean())


<font face="Times New Roman">

### 5.5 Visualización Gráfica de Degradación (HMEQ)

Se ilustra comparativamente el comportamiento de ambos algoritmos en todo el espacio experimental:
* **Líneas continuas**: XGBoost.
* **Líneas punteadas**: Random Forest.

</font>


In [ ]:
FIG_DIR_ROB = 'figures_hmeq'
os.makedirs(FIG_DIR_ROB, exist_ok=True)

colores = {'Base': 'gray', 'RUS': 'red', 'ROS': 'green'}
markers_xgb = {'Base': 'o', 'RUS': 's', 'ROS': '^'}
markers_rf = {'Base': 'x', 'RUS': '+', 'ROS': 'd'}

plots_rob = [
    ('SHAP CV (XGB)', 'SHAP CV (RF)', 'Degradación CV de SHAP (HMEQ)', 'Promedio CV (Menor = Más estable)', 'shap_cv'),
    ('LIME CV (XGB)', 'LIME CV (RF)', 'Degradación CV de LIME (HMEQ)', 'Promedio CV (Menor = Más estable)', 'lime_cv'),
    ('SHAP SRA (XGB)', 'SHAP SRA (RF)', 'Degradación SRA (d=1) de SHAP (HMEQ)', 'SRA d=1 (Menor = Más estable)', 'shap_sra'),
    ('LIME SRA (XGB)', 'LIME SRA (RF)', 'Degradación SRA (d=1) de LIME (HMEQ)', 'SRA d=1 (Menor = Más estable)', 'lime_sra'),
    ('AUC (XGB)', 'AUC (RF)', 'Capacidad Predictiva: AUC (HMEQ)', 'AUC (Mayor = Mejor)', 'auc'),
    ('F1-Score (XGB)', 'F1-Score (RF)', 'Capacidad Predictiva: F1-Score (HMEQ)', 'F1-Score (Mayor = Mejor)', 'f1'),
]


# Consolidación gráfica: Estandarizar ejes Y para comparabilidad
max_cv_rob = df_rob[['SHAP CV (XGB)', 'SHAP CV (RF)', 'LIME CV (XGB)', 'LIME CV (RF)']].max().max() * 1.1
max_sra_rob = df_rob[['SHAP SRA (XGB)', 'SHAP SRA (RF)', 'LIME SRA (XGB)', 'LIME SRA (RF)']].max().max() * 1.1

for col_xgb, col_rf, title, ylabel, fname in plots_rob:

    fig, ax = plt.subplots(figsize=(10, 6))
    for esc in escenarios_rob:
        s_df = df_rob[df_rob['Escenario'] == esc]
        x_axis = s_df['Imbalance (%)']
        ax.plot(x_axis, s_df[col_xgb], label=f"{esc} (XGB)", color=colores[esc], marker=markers_xgb[esc], linestyle='-')
        ax.plot(x_axis, s_df[col_rf],  label=f"{esc} (RF)",  color=colores[esc], marker=markers_rf[esc],  linestyle='--')
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Nivel de Desbalanceo (%)', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(x_axis)
    ax.set_xticklabels([f"{x:.1f}%" for x in x_axis], rotation=45)
    if 'cv' in fname:
        ax.set_ylim(0, max_cv_rob)
    elif 'sra' in fname:
        ax.set_ylim(0, max_sra_rob)
    ax.legend(title='Escenario', fontsize='small', loc='best')
    ax.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    save_path = os.path.join(FIG_DIR_ROB, f'hmeq_{fname}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardada: {save_path}")


<font face="Times New Roman">

### 5.6 Pruebas Estadísticas Finales (Robustez HMEQ)

Al igual que con la primera base de datos, validaremos primero el supuesto de normalidad con la prueba de Shapiro-Wilk para el coeficiente de variación (CV) y el SRA (d=1) de SHAP. Al confirmar el no cumplimiento de la normalidad, utilizaremos el test no paramétrico de Friedman.

</font>


In [ ]:
print("=== Análisis Estadístico en Entorno HMEQ ===")
escenarios_list_rob = ['Base', 'RUS', 'ROS']

# --- Bloque 1: CV de SHAP (XGBoost) ---
matriz_cv_rob = np.array([[cv_shap_xgb_rob[lvl][esc] for esc in escenarios_list_rob] for lvl in niveles_desbalance_rob])
mask_cv = ~np.isnan(matriz_cv_rob).any(axis=1)
matriz_cv_valida = matriz_cv_rob[mask_cv]

print("1. Normalidad para Estabilidad SHAP (CV XGB) via Shapiro-Wilk:")
if len(matriz_cv_valida) > 2:
    p_vals = [shapiro(matriz_cv_valida[:, j])[1] for j in range(3)]
    for name, p in zip(escenarios_list_rob, p_vals):
        print(f"   - {name} p-value: {p:.4e}")
    if any(p < 0.05 for p in p_vals):
        print("   -> Rechazamos normalidad. Procedemos con Friedman.")
    else:
        print("   -> No rechazamos normalidad, pero procedemos con Friedman por consistencia no paramétrica.")
    stat, p = friedmanchisquare(matriz_cv_valida[:, 0], matriz_cv_valida[:, 1], matriz_cv_valida[:, 2])
    print(f"   -> Friedman Test (HMEQ SHAP CV XGB) -> Statistic: {stat:.4f}, p-value: {p:.4e}\n")

# --- Bloque 2: SRA de SHAP (XGBoost) ---
matriz_sra_rob = np.array([[sra_xgb_rob[lvl][esc] for esc in escenarios_list_rob] for lvl in niveles_desbalance_rob])
mask_sra = ~np.isnan(matriz_sra_rob).any(axis=1)
matriz_sra_valida = matriz_sra_rob[mask_sra]

print("2. Normalidad para Concordancia SHAP SRA (d=1) via Shapiro-Wilk:")
if len(matriz_sra_valida) > 2:
    p_vals_sra = [shapiro(matriz_sra_valida[:, j])[1] for j in range(3)]
    for name, p in zip(escenarios_list_rob, p_vals_sra):
        print(f"   - {name} p-value: {p:.4e}")
    if any(p < 0.05 for p in p_vals_sra):
        print("   -> Rechazamos normalidad. Procedemos con Friedman.")
    else:
        print("   -> No rechazamos normalidad, pero procedemos con Friedman por consistencia no paramétrica.")
    stat_s, p_s = friedmanchisquare(matriz_sra_valida[:, 0], matriz_sra_valida[:, 1], matriz_sra_valida[:, 2])
    print(f"   -> Friedman Test (HMEQ SHAP SRA XGB) -> Statistic: {stat_s:.4f}, p-value: {p_s:.4e}\n")


<font face="Times New Roman">

### 5.7 Análisis de Resultados Estadísticos (HMEQ)

**1. Verificación de Supuestos:**
Las pruebas de Shapiro-Wilk ejecutadas sobre los tres escenarios (Base, RUS, ROS) para ambas métricas —CV de SHAP y SRA (d=1) de SHAP— demostraron un incumplimiento del supuesto de normalidad. Al violarse este supuesto, se descartó el uso de ANOVA de medidas repetidas y se procedió formalmente con la prueba no paramétrica de Friedman.

**2. Resultados de Friedman:**
Ambas pruebas arrojaron p-values inferiores a $0.001$, estrictamente por debajo del nivel de significancia $\alpha = 0.05$.

**Interpretación:**
Se confirma que **el nivel de desbalanceo y la técnica de remuestreo (RUS/ROS) afectan de forma estadísticamente significativa** tanto la magnitud de las atribuciones SHAP (CV) como la consistencia del ordenamiento de las variables (SRA d=1), replicando los hallazgos del experimento principal con German Credit Risk y validando así la robustez externa de los resultados.

</font>


<font face="Times New Roman">

### Descarga de gráficos de resultados

In [ ]:
import shutil
from google.colab import files

# 1. Comprimir la carpeta de German Credit
shutil.make_archive('figures_german_credit', 'zip', 'figures_german_credit')

# 2. Comprimir la carpeta de HMEQ (Robustez)
shutil.make_archive('figures_hmeq', 'zip', 'figures_hmeq')

# 3. Descargar los archivos ZIP
files.download('figures_german_credit.zip')
files.download('figures_hmeq.zip')
